In [10]:
import numpy as np
import pickle
import torch
# import lightgbm as lgb
from torchmetrics.regression import MeanSquaredError
from torchmetrics.functional.image import peak_signal_noise_ratio
from torchmetrics.functional.image import structural_similarity_index_measure

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt
import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
from typing import Tuple, List

In [ ]:
with open('/app/data/processed/cmip6-cmip6/preds_factor2/train_ens_data_unet_42.pkl', 'rb') as f:
    unet_42 = pickle.load(f)

with open('/app/data/processed/cmip6-cmip6/preds_factor2/train_ens_data_unet_123.pkl', 'rb') as f:
    unet_123 = pickle.load(f)

with open('/app/data/processed/cmip6-cmip6/preds_factor2/train_ens_data_unet_777.pkl', 'rb') as f:
    unet_777 = pickle.load(f)

with open('/app/data/processed/cmip6-cmip6/preds_factor2/y_true_train_ens_data.pkl', 'rb') as f:
    train_y_true = pickle.load(f)

# Val
with open('/app/data/processed/cmip6-cmip6/preds_factor2/val_ens_data.pkl', 'rb') as f:
    val_data = pickle.load(f)

with open('/app/data/processed/cmip6-cmip6/preds_factor2/y_true_val_ens_data.pkl', 'rb') as f:
    val_y_true = pickle.load(f)

# Test
with open('/app/data/processed/cmip6-cmip6/preds_factor2/test_ens_data.pkl', 'rb') as f:
    test_data = pickle.load(f)

with open('/app/data/processed/cmip6-cmip6/preds_factor2/y_true_test_ens_data.pkl', 'rb') as f:
    test_y_true = pickle.load(f)

train_data = torch.stack([train_data[model] for model in ['resnet_42', 'unet_42', 'ynet_42', 'vit_42', 'hat_42', 'esrgan_42', 'diffusion_42']], dim=1)

In [6]:
(
    MeanSquaredError(squared=False)(unet_42, train_y_true),
    MeanSquaredError(squared=False)(unet_123, train_y_true),
    MeanSquaredError(squared=False)(unet_777, train_y_true)
)

(tensor(0.1069), tensor(0.1069), tensor(0.1037))

In [9]:
MeanSquaredError(squared=False)((unet_42 + unet_123 + unet_777)/3, train_y_true)

tensor(0.1040)

In [ ]:
    # Дополнительные улучшения модели
class EnhancedEnsemble_3(nn.Module):
    """Улучшенная версия с несколькими слоями"""
    def __init__(self, n_models: int = 3, n_channels: int = 4, hidden_channels: int = 16):
        super().__init__()
        
        self.network = nn.Sequential(
            # Первый слой: объединение моделей
            nn.Conv2d(n_models * n_channels, hidden_channels, kernel_size=1),
            nn.ReLU(),
            nn.BatchNorm2d(hidden_channels),
            
            # Второй слой: дополнительная обработка
            nn.Conv2d(hidden_channels, hidden_channels, kernel_size=1),
            nn.ReLU(),
            nn.BatchNorm2d(hidden_channels),
            
            # Третий слой: дополнительная обработка
            nn.Conv2d(hidden_channels, hidden_channels, kernel_size=1),
            nn.ReLU(),
            nn.BatchNorm2d(hidden_channels),
            
            # Финальный слой: проекция к нужному числу каналов
            nn.Conv2d(hidden_channels, n_channels, kernel_size=1)
        )
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        batch_size = x.shape[0]
        height, width = x.shape[-2], x.shape[-1]
        x_reshaped = x.view(batch_size, -1, height, width)
        return self.network(x_reshaped)
    

def train_mlp_ensemble(model,X_train_tensor,
                       X_val_tensor,
                       y_train_tensor,
                       y_val_tensor,
                       n_models,
                       epochs=100,
                       batch_size=8,
                       learning_rate=0.001
                       ):
    """
    Обучение MLP ансамбля
    
    Args:
        X: входные данные формы (n_samples, n_models, 4, 192, 384)
        y: целевые данные формы (n_samples, 4, 192, 384)
        n_models: количество моделей в ансамбле
        epochs: количество эпох
        batch_size: размер батча
        learning_rate: скорость обучения
    """
    
    # Создание DataLoader
    train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
    val_dataset = TensorDataset(X_val_tensor, y_val_tensor)
    
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    
    # Инициализация модели
    device = 'cuda:2'#torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model#EnhancedEnsemble_3(n_models=7).to(device)
    
    # Функция потерь и оптимизатор
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)
    
    # Для сохранения истории обучения
    train_losses = []
    val_losses = []
    
    print("Начинаем обучение MLP ансамбля...")
    print(f"Устройство: {device}")
    print(f"Размер тренировочной выборки: {len(X_train_tensor)}")
    print(f"Размер валидационной выборки: {len(X_val_tensor)}")
    
    for epoch in tqdm.tqdm(range(epochs)):
        # Тренировка
        model.train()
        train_loss = 0.0
        
        for batch_X, batch_y in train_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            
            optimizer.zero_grad()
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item() * batch_X.size(0)
        
        # Валидация
        model.eval()
        val_loss = 0.0
        
        with torch.no_grad():
            for batch_X, batch_y in val_loader:
                batch_X, batch_y = batch_X.to(device), batch_y.to(device)
                outputs = model(batch_X)
                loss = criterion(outputs, batch_y)
                val_loss += loss.item() * batch_X.size(0)
        
        # Средние потери
        train_loss = train_loss / len(train_loader.dataset)
        val_loss = val_loss / len(val_loader.dataset)
        
        train_losses.append(train_loss)
        val_losses.append(val_loss)
        
        scheduler.step(val_loss)
        
        if (epoch + 1) % 10 == 0:
            print(f'Epoch [{epoch+1}/{epochs}], '
                  f'Train Loss: {train_loss:.6f}, '
                  f'Val Loss: {val_loss:.6f}, '
                  f'LR: {optimizer.param_groups[0]["lr"]:.2e}')
    
        # График обучения
        plt.figure(figsize=(10, 5))
        plt.plot(train_losses, label='Train Loss')
        plt.plot(val_losses, label='Validation Loss')
        plt.xlabel('Epoch')
        plt.ylabel('Loss')
        plt.title('Training History')
        plt.legend()
        plt.yscale('log')
        plt.show()
    
    return model, train_losses, val_losses


# Обучение модели
model, train_losses, val_losses = train_mlp_ensemble(
    train_data,
    val_data,
    train_y_true,
    val_y_true,
    n_models=3, 
    epochs=20, 
    batch_size=8,
    learning_rate=0.0005
)


tensor([[[[-1.3084, -1.3193, -1.3200,  ..., -1.3394, -1.3224, -1.3058],
          [-1.3359, -1.3301, -1.3200,  ..., -1.3288, -1.3239, -1.3353],
          [-1.3281, -1.3297, -1.3256,  ..., -1.3279, -1.3322, -1.3346],
          ...,
          [-1.4157, -1.4146, -1.4162,  ..., -1.4003, -1.4074, -1.4028],
          [-1.4075, -1.4181, -1.4173,  ..., -1.4089, -1.4002, -1.4008],
          [-1.4289, -1.4403, -1.4303,  ..., -1.4316, -1.4267, -1.4369]],

         [[-1.2570, -1.2680, -1.2719,  ..., -1.2597, -1.2562, -1.2480],
          [-1.3787, -1.3618, -1.3531,  ..., -1.4119, -1.4045, -1.3959],
          [-1.5252, -1.5029, -1.4909,  ..., -1.5522, -1.5431, -1.5361],
          ...,
          [-0.4143, -0.4139, -0.4099,  ..., -0.4365, -0.4269, -0.4217],
          [-0.5504, -0.5512, -0.5428,  ..., -0.5511, -0.5546, -0.5649],
          [-0.6179, -0.6519, -0.6285,  ..., -0.6250, -0.6180, -0.6423]],

         [[ 0.2201,  0.2383,  0.2508,  ...,  0.1365,  0.1616,  0.1903],
          [ 0.2925,  0.3174,  